# DS3/GSE182109 — sample-aware post-QC Scrublet diagnostic

This notebook reproduces the sample-aware Scrublet screening recorded for
DS3 while keeping the large post-QC matrix on disk.

Memory-safe execution strategy:

- the full `GSE182109_postQC.h5ad` file is opened directly with HDF5;
- Scrublet is run independently within each sample;
- only the CSR raw-count rows for the current sample chunk are loaded into RAM;
- low-information genes/cells and highly variable genes are selected before the Scrublet core;
- `mean_center=False` keeps dimensionality reduction sparse-compatible;
- completed sample chunks are cached deterministically;
- the original H5AD is copied on disk and only `obs` and `uns` are replaced,
  so the full expression matrix and `counts` layer are never loaded together.

Recorded historical result:

- 2,927 predicted doublets;
- 215,146 predicted singlets;
- 662 cells not checked because two samples contained fewer than 500
  post-QC cells.

These calls are retained as a diagnostic and are **not used to remove cells
from the main analysis**, whose DS3 input remains 218,735 cells.

Input:

`data/processed/DS3_GSE182109/GSE182109_postQC.h5ad`

Output:

`data/processed/DS3_GSE182109/GSE182109_postQC_scrublet_checked.h5ad`


## Runtime patch v4

This version reads the CSR count layer directly from HDF5, performs the same Scanpy gene filtering and default HVG selection **before** calling the Scrublet core, and caches each completed sample. It also resolves the installed Scanpy version's RNG keyword and explicitly simulates doublets from `adata.X` (`layer=None`) rather than looking for a nonexistent layer named `"X"`.


In [1]:
import gc
import hashlib
import importlib.util
import inspect
import json
import os
import re
import shutil
import sys
import traceback
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import anndata as ad
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.utils.sparsefuncs import inplace_csr_row_scale

try:
    from anndata.io import read_elem
except ImportError:
    # Compatibility with older AnnData releases.
    from anndata._io.specs import read_elem

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)


Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
anndata: 0.12.7
scanpy: 1.11.5


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_34888\174754156.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_34888\174754156.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS3_GSE182109"
        / "GSE182109_postQC.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_SCRUBLET_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS3_GSE182109"
        / "GSE182109_postQC_scrublet_checked.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT
    / "results"
    / "qc"
    / "DS3_GSE182109"
    / "00c_doublet_diagnostic"
)
FIGURE_DIR = AUDIT_DIR / "figures"
CACHE_DIR = AUDIT_DIR / "sample_cache_v4"

EXPECTED_INPUT_SHAPE = (218_735, 30_284)
EXPECTED_PREDICTED_DOUBLETS = 2_927
EXPECTED_PREDICTED_SINGLETS = 215_146
EXPECTED_NOT_CHECKED = 662

SAMPLE_COLUMN = "sample"
MIN_CELLS_PER_SAMPLE = 500
MAX_CELLS_PER_RUN = int(
    os.environ.get("GSE182109_SCRUBLET_MAX_CELLS", "40000")
)
EXPECTED_DOUBLET_RATE = 0.05
SIM_DOUBLET_RATIO = 1.0
N_PCS = 20
RANDOM_SEED = 0

# These reproduce the filtering done internally by scanpy.pp.scrublet,
# but before Scanpy creates several full-size AnnData copies.
SCRUBLET_MIN_CELLS_PER_GENE = 3
SCRUBLET_MIN_GENES_PER_CELL = 3
GENE_NNZ_CHUNK = int(
    os.environ.get("GSE182109_GENE_NNZ_CHUNK", "1000000")
)
RUNTIME_PATCH_VERSION = "ds3-scrublet-direct-hdf5-prehvg-v4"

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Post-QC input not found: {INPUT_H5AD}\n"
        "Run 00b_ds3_quality_control.ipynb first or set "
        "GSE182109_POSTQC_H5AD."
    )

if importlib.util.find_spec("scrublet") is None:
    raise ImportError(
        "The 'scrublet' package is required by scanpy.pp.scrublet. "
        "Install the repository environment before running this notebook."
    )

if MAX_CELLS_PER_RUN < MIN_CELLS_PER_SAMPLE:
    raise ValueError(
        "MAX_CELLS_PER_RUN must be at least MIN_CELLS_PER_SAMPLE."
    )

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)
print("Maximum cells per Scrublet run:", MAX_CELLS_PER_RUN)
print("Runtime patch:", RUNTIME_PATCH_VERSION)


Input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS3_GSE182109\GSE182109_postQC.h5ad
Output: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS3_GSE182109\GSE182109_postQC_scrublet_checked.h5ad
Maximum cells per Scrublet run: 40000
Runtime patch: ds3-scrublet-direct-hdf5-prehvg-v4


In [3]:
def iter_array_chunks(array, chunk_size: int):
    for start in range(0, len(array), chunk_size):
        yield array[start : start + chunk_size]


def assert_raw_counts_chunked(matrix, label: str, chunk_size: int = 1_000_000):
    """Validate sparse raw counts without allocating nnz-sized temporaries."""
    if not sp.isspmatrix_csr(matrix):
        raise TypeError(
            f"{label} must be CSR; observed {type(matrix).__name__}."
        )
    if matrix.nnz == 0:
        raise ValueError(f"{label} contains no nonzero values.")

    for data_chunk in iter_array_chunks(matrix.data, chunk_size):
        if not np.isfinite(data_chunk).all():
            raise ValueError(f"{label} contains non-finite values.")
        if np.any(data_chunk < 0):
            raise ValueError(f"{label} contains negative values.")

        # Counts are float32 on disk. Validate integrality in bounded chunks.
        rounded = np.rint(data_chunk)
        if not np.array_equal(data_chunk, rounded):
            raise ValueError(f"{label} is not an integer raw-count matrix.")


def _add_supported_seed_parameter(
    requested: dict,
    function,
    function_label: str,
) -> tuple[dict, str]:
    """Attach RANDOM_SEED using the keyword exposed by this Scanpy build."""
    signature = inspect.signature(function)
    supported = set(signature.parameters)

    # Scanpy releases have used all three names across versions.
    for keyword in ("random_seed", "random_state", "rng"):
        if keyword in supported:
            requested[keyword] = RANDOM_SEED
            return requested, keyword

    raise RuntimeError(
        f"{function_label} exposes none of random_seed, random_state, or rng. "
        f"Observed signature: {signature}"
    )


def core_scrublet_parameters(n_prin_comps: int) -> dict:
    requested = {
        "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
        "n_prin_comps": int(n_prin_comps),
        # Critical runtime patch: use sparse TruncatedSVD, not dense PCA.
        "mean_center": False,
        "normalize_variance": True,
        "log_transform": False,
        "use_approx_neighbors": True,
        "copy": False,
    }

    requested, _ = _add_supported_seed_parameter(
        requested,
        sc.pp.scrublet,
        "scanpy.pp.scrublet",
    )

    signature = inspect.signature(sc.pp.scrublet)
    supported = set(signature.parameters)
    missing = sorted(set(requested) - supported)
    if missing:
        raise RuntimeError(
            "This Scanpy version does not support required Scrublet "
            f"parameters: {missing}. Observed signature: {signature}"
        )
    return requested


def simulation_parameters() -> dict:
    # scrublet_simulate_doublets interprets layer as an AnnData layer name.
    # The working raw counts are stored in observed.X, so layer must be None
    # (or omitted). Passing the string "X" incorrectly asks for
    # observed.layers["X"] and raises KeyError.
    requested = {
        "sim_doublet_ratio": SIM_DOUBLET_RATIO,
        "synthetic_doublet_umi_subsampling": 1.0,
    }

    signature = inspect.signature(sc.pp.scrublet_simulate_doublets)
    supported = set(signature.parameters)
    if "layer" in supported:
        requested["layer"] = None

    requested, _ = _add_supported_seed_parameter(
        requested,
        sc.pp.scrublet_simulate_doublets,
        "scanpy.pp.scrublet_simulate_doublets",
    )

    missing = sorted(set(requested) - supported)
    if missing:
        raise RuntimeError(
            "This Scanpy version does not support required doublet "
            f"simulation parameters: {missing}. Observed signature: {signature}"
        )
    return requested


def validate_scrublet_api_compatibility() -> None:
    """Fail early, before loading any sample matrix, on API incompatibility."""
    simulation = simulation_parameters()
    core = core_scrublet_parameters(max(2, min(N_PCS, 5)))

    if simulation.get("layer", None) is not None:
        raise AssertionError(
            "Doublet simulation must read observed.X, so layer must be None."
        )

    print("Scrublet API compatibility check passed.")
    print(
        "  scrublet_simulate_doublets signature:",
        inspect.signature(sc.pp.scrublet_simulate_doublets),
    )
    print("  simulation parameters:", simulation)
    print("  scrublet signature:", inspect.signature(sc.pp.scrublet))
    print("  core RNG parameter:", [
        key for key in ("random_seed", "random_state", "rng") if key in core
    ][0])


validate_scrublet_api_compatibility()


def decode_h5_attr(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return value


def stored_matrix_shape(node) -> tuple[int, int]:
    """Return a dense or sparse HDF5 matrix shape without reading its data."""
    if isinstance(node, h5py.Dataset):
        return tuple(int(value) for value in node.shape)

    if not isinstance(node, h5py.Group):
        raise TypeError(f"Unsupported HDF5 matrix node: {type(node).__name__}")

    shape = node.attrs.get("shape")
    if shape is None:
        raise KeyError(f"Sparse matrix group {node.name!r} has no shape attribute.")

    return tuple(int(value) for value in shape)


def stored_sparse_encoding(group: h5py.Group) -> str:
    encoding = decode_h5_attr(group.attrs.get("encoding-type", ""))
    return str(encoding)


def contiguous_runs(sorted_positions: np.ndarray):
    """Yield inclusive-exclusive contiguous row runs."""
    if sorted_positions.size == 0:
        return

    breaks = np.flatnonzero(np.diff(sorted_positions) != 1) + 1
    for block in np.split(sorted_positions, breaks):
        yield int(block[0]), int(block[-1]) + 1


def load_csr_rows(counts_group: h5py.Group, row_positions):
    """Read selected CSR rows directly from HDF5 without AnnData backed I/O."""
    row_positions = np.asarray(row_positions, dtype=np.int64)

    if row_positions.ndim != 1 or row_positions.size == 0:
        raise ValueError("row_positions must be a non-empty 1D array.")
    if np.unique(row_positions).size != row_positions.size:
        raise ValueError("row_positions contains duplicate rows.")

    encoding = stored_sparse_encoding(counts_group)
    if encoding != "csr_matrix":
        raise RuntimeError(
            "The memory-safe reader requires layers['counts'] to be CSR. "
            f"Observed encoding: {encoding!r}."
        )

    n_obs, n_vars = stored_matrix_shape(counts_group)
    if row_positions.min() < 0 or row_positions.max() >= n_obs:
        raise IndexError("row_positions contains an out-of-range row index.")

    sorted_positions = np.sort(row_positions)
    blocks = []

    for row_start, row_end in contiguous_runs(sorted_positions):
        stored_indptr = np.asarray(
            counts_group["indptr"][row_start : row_end + 1],
            dtype=np.int64,
        )
        data_start = int(stored_indptr[0])
        data_end = int(stored_indptr[-1])

        data = np.asarray(
            counts_group["data"][data_start:data_end],
            dtype=np.float32,
        )
        indices = np.asarray(
            counts_group["indices"][data_start:data_end],
            dtype=np.int32,
        )
        local_indptr = stored_indptr - data_start

        block = sp.csr_matrix(
            (data, indices, local_indptr),
            shape=(row_end - row_start, n_vars),
            dtype=np.float32,
        )
        blocks.append(block)

    counts = blocks[0] if len(blocks) == 1 else sp.vstack(blocks, format="csr")

    # The H5AD CSR source is already canonical and zero-free. Avoid
    # sum_duplicates()/eliminate_zeros(), which allocate nnz-sized temporaries.
    if counts.shape != (len(sorted_positions), n_vars):
        raise AssertionError(
            "Direct CSR row reader returned an unexpected shape: "
            f"{counts.shape}"
        )

    return sorted_positions, counts


def column_nnz_chunked(csr: sp.csr_matrix, chunk_size: int) -> np.ndarray:
    """Count expressing cells per gene without one huge np.bincount input."""
    result = np.zeros(csr.shape[1], dtype=np.int64)
    for index_chunk in iter_array_chunks(csr.indices, chunk_size):
        result += np.bincount(
            index_chunk,
            minlength=csr.shape[1],
        )
    return result


def select_scanpy_default_hvgs(
    raw_counts: sp.csr_matrix,
    raw_var_names: pd.Index,
    label: str,
):
    """Reproduce Scanpy Scrublet preprocessing with bounded peak memory.

    scanpy.pp.scrublet normally creates several full-size AnnData copies before
    HVG selection. Here we create only one normalized-log1p working matrix,
    derive the installed Scanpy version's default HVG mask, delete the working
    matrix, and finally subset the raw counts directly to HVGs.
    """
    row_nnz = np.diff(raw_counts.indptr)
    cell_keep = row_nnz >= SCRUBLET_MIN_GENES_PER_CELL

    if not cell_keep.all():
        working_raw = raw_counts[cell_keep, :].copy()
    else:
        working_raw = raw_counts

    gene_nnz = column_nnz_chunked(
        working_raw,
        chunk_size=GENE_NNZ_CHUNK,
    )
    gene_keep = gene_nnz >= SCRUBLET_MIN_CELLS_PER_GENE
    n_genes_after_min_cells = int(gene_keep.sum())

    if n_genes_after_min_cells <= N_PCS:
        raise RuntimeError(
            f"{label}: only {n_genes_after_min_cells} genes remain after "
            "min_cells=3 filtering."
        )

    filtered_var_names = raw_var_names[gene_keep]

    # This is the only full-size working copy. The original raw matrix remains
    # untouched for the final direct subset to selected HVGs.
    normalized_log = working_raw[:, gene_keep].copy()

    cell_totals = np.asarray(normalized_log.sum(axis=1)).ravel()
    positive_totals = cell_totals[cell_totals > 0]
    if positive_totals.size == 0:
        raise ValueError(f"{label}: all filtered cells have zero total counts.")

    # scanpy.pp.normalize_total(target_sum=None) uses the median cell total.
    target_sum = float(np.median(positive_totals))
    scale = np.divide(
        target_sum,
        cell_totals,
        out=np.zeros_like(cell_totals, dtype=np.float32),
        where=cell_totals > 0,
    )

    inplace_csr_row_scale(
        normalized_log,
        scale.astype(np.float32, copy=False),
    )
    np.log1p(normalized_log.data, out=normalized_log.data)

    hvg_adata = ad.AnnData(
        X=normalized_log,
        var=pd.DataFrame(index=filtered_var_names.copy()),
    )

    # No explicit cutoffs: use the installed Scanpy version's same defaults
    # that scanpy.pp.scrublet calls internally.
    sc.pp.highly_variable_genes(hvg_adata)

    if "highly_variable" not in hvg_adata.var:
        raise RuntimeError(f"{label}: HVG selection did not create a mask.")

    hvg_within_filtered = hvg_adata.var[
        "highly_variable"
    ].to_numpy(dtype=bool)
    n_hvg = int(hvg_within_filtered.sum())

    del hvg_adata
    del normalized_log
    gc.collect()

    if n_hvg <= N_PCS:
        raise RuntimeError(
            f"{label}: only {n_hvg} highly variable genes were selected; "
            f"more than {N_PCS} are required."
        )

    full_hvg_mask = np.zeros(raw_counts.shape[1], dtype=bool)
    full_hvg_mask[np.flatnonzero(gene_keep)[hvg_within_filtered]] = True

    source_for_subset = working_raw
    hvg_counts = source_for_subset[:, full_hvg_mask].copy()
    hvg_var_names = raw_var_names[full_hvg_mask]

    if working_raw is not raw_counts:
        del working_raw
    gc.collect()

    return (
        cell_keep,
        hvg_counts,
        hvg_var_names,
        n_genes_after_min_cells,
        n_hvg,
    )

def run_one_scrublet_chunk(
    counts_group,
    obs_names,
    var_names,
    row_positions,
    sample_id,
    chunk_label,
):
    sorted_positions, raw_counts = load_csr_rows(
        counts_group,
        row_positions,
    )

    label = f"{sample_id}, chunk {chunk_label}"
    assert_raw_counts_chunked(raw_counts, label)

    (
        cell_keep,
        hvg_counts,
        hvg_var_names,
        n_genes_after_min_cells,
        n_hvg,
    ) = select_scanpy_default_hvgs(
        raw_counts,
        var_names,
        label,
    )
    del raw_counts
    gc.collect()

    kept_positions = sorted_positions[cell_keep]
    chunk_obs_names = obs_names[kept_positions].copy()

    n_pcs = min(
        N_PCS,
        hvg_counts.shape[0] - 1,
        hvg_counts.shape[1] - 1,
    )
    if n_pcs < 2:
        raise RuntimeError(f"{label}: too few cells or HVGs for SVD.")

    observed = ad.AnnData(
        X=hvg_counts,
        obs=pd.DataFrame(index=chunk_obs_names),
        var=pd.DataFrame(index=hvg_var_names.copy()),
    )

    simulated = None
    try:
        simulated = sc.pp.scrublet_simulate_doublets(
            observed,
            **simulation_parameters(),
        )

        sc.pp.normalize_total(observed, target_sum=1e6)
        sc.pp.normalize_total(simulated, target_sum=1e6)

        sc.pp.scrublet(
            observed,
            adata_sim=simulated,
            **core_scrublet_parameters(n_pcs),
        )

        required_columns = {"doublet_score", "predicted_doublet"}
        missing = required_columns - set(observed.obs.columns)
        if missing:
            raise RuntimeError(
                f"Scrublet did not create required columns: {missing}"
            )

        scores = pd.to_numeric(
            observed.obs["doublet_score"],
            errors="raise",
        ).to_numpy(dtype=np.float64)
        predictions = (
            observed.obs["predicted_doublet"]
            .astype(bool)
            .to_numpy()
        )

        if not np.isfinite(scores).all():
            raise ValueError(f"{label}: Scrublet returned non-finite scores.")

        diagnostics = {
            "input_cells": int(len(sorted_positions)),
            "checked_cells": int(len(kept_positions)),
            "genes_after_min_cells": n_genes_after_min_cells,
            "highly_variable_genes": n_hvg,
            "n_prin_comps": int(n_pcs),
        }

        return kept_positions, scores, predictions, diagnostics

    finally:
        if simulated is not None:
            del simulated
        del observed
        del hvg_counts
        gc.collect()


def cache_path_for(sample_id: str, chunk_label: str) -> Path:
    digest = hashlib.sha256(
        f"{sample_id}|{chunk_label}|{RUNTIME_PATCH_VERSION}".encode("utf-8")
    ).hexdigest()[:16]
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(sample_id))[:80]
    return CACHE_DIR / f"{safe_name}__{chunk_label}__{digest}.npz"


def save_chunk_cache(path, positions, scores, predictions, diagnostics):
    np.savez_compressed(
        path,
        positions=np.asarray(positions, dtype=np.int64),
        scores=np.asarray(scores, dtype=np.float64),
        predictions=np.asarray(predictions, dtype=np.bool_),
        diagnostics_json=np.asarray(
            json.dumps(diagnostics, sort_keys=True),
            dtype="U",
        ),
        runtime_patch=np.asarray(RUNTIME_PATCH_VERSION, dtype="U"),
    )


def load_chunk_cache(path, expected_positions):
    if not path.exists():
        return None

    with np.load(path, allow_pickle=False) as cached:
        if str(cached["runtime_patch"].item()) != RUNTIME_PATCH_VERSION:
            return None
        positions = cached["positions"].astype(np.int64, copy=False)
        if not np.array_equal(positions, np.asarray(expected_positions, dtype=np.int64)):
            return None
        scores = cached["scores"].astype(np.float64, copy=False)
        predictions = cached["predictions"].astype(bool, copy=False)
        diagnostics = json.loads(cached["diagnostics_json"].item())

    if len(scores) != len(positions) or len(predictions) != len(positions):
        return None

    return positions, scores, predictions, diagnostics


Scrublet API compatibility check passed.
  scrublet_simulate_doublets signature: (adata: 'AnnData', *, layer: 'str | None' = None, sim_doublet_ratio: 'float' = 2.0, synthetic_doublet_umi_subsampling: 'float' = 1.0, random_seed: '_LegacyRandom' = 0) -> 'AnnData'
  simulation parameters: {'sim_doublet_ratio': 1.0, 'synthetic_doublet_umi_subsampling': 1.0, 'layer': None, 'random_seed': 0}
  scrublet signature: (adata: 'AnnData', adata_sim: 'AnnData | None' = None, *, batch_key: 'str | None' = None, sim_doublet_ratio: 'float' = 2.0, expected_doublet_rate: 'float' = 0.05, stdev_doublet_rate: 'float' = 0.02, synthetic_doublet_umi_subsampling: 'float' = 1.0, knn_dist_metric: '_Metric | _MetricFn' = 'euclidean', normalize_variance: 'bool' = True, log_transform: 'bool' = False, mean_center: 'bool' = True, n_prin_comps: 'int' = 30, use_approx_neighbors: 'bool | None' = None, get_doublet_neighbor_parents: 'bool' = False, n_neighbors: 'int | None' = None, threshold: 'float | None' = None, verbose:

In [4]:
print("Opening DS3 H5AD with direct HDF5 access")

# AnnData backed mode still materializes sparse layers in the installed
# version. Open the container directly and read only metadata-sized groups.
input_h5 = h5py.File(INPUT_H5AD, "r")

try:
    if "X" not in input_h5:
        raise KeyError("The input H5AD is missing /X.")
    if "layers" not in input_h5 or "counts" not in input_h5["layers"]:
        raise KeyError("The post-QC object is missing /layers/counts.")
    if "obs" not in input_h5 or "var" not in input_h5:
        raise KeyError("The input H5AD is missing /obs or /var metadata.")

    observed_shape = stored_matrix_shape(input_h5["X"])
    counts_group = input_h5["layers"]["counts"]
    counts_shape = stored_matrix_shape(counts_group)
    counts_encoding = stored_sparse_encoding(counts_group)

    if observed_shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            f"Expected input shape {EXPECTED_INPUT_SHAPE}, "
            f"observed {observed_shape}"
        )
    if counts_shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            "The raw-count layer has an unexpected shape: "
            f"{counts_shape}"
        )
    if counts_encoding != "csr_matrix":
        raise RuntimeError(
            "Direct row-wise Scrublet requires a CSR counts layer. "
            f"Observed encoding: {counts_encoding!r}"
        )

    obs_output = read_elem(input_h5["obs"])
    var_output = read_elem(input_h5["var"])
    uns_output = (
        deepcopy(read_elem(input_h5["uns"]))
        if "uns" in input_h5
        else {}
    )

    if len(obs_output) != EXPECTED_INPUT_SHAPE[0]:
        raise AssertionError("The /obs table has an unexpected row count.")
    if len(var_output) != EXPECTED_INPUT_SHAPE[1]:
        raise AssertionError("The /var table has an unexpected row count.")
    if SAMPLE_COLUMN not in obs_output.columns:
        raise KeyError(f"obs[{SAMPLE_COLUMN!r}] is required.")
    if not obs_output.index.is_unique:
        raise AssertionError("Cell identifiers are not unique.")
    if not var_output.index.is_unique:
        raise AssertionError("Gene identifiers are not unique.")

    obs_names = pd.Index(obs_output.index.astype(str), dtype="object")
    var_names = pd.Index(var_output.index.astype(str), dtype="object")

except Exception:
    input_h5.close()
    raise

for column in (
    "doublet_score",
    "predicted_doublet",
    "doublet_check_status",
):
    if column in obs_output.columns:
        del obs_output[column]

doublet_scores = np.full(
    EXPECTED_INPUT_SHAPE[0],
    np.nan,
    dtype=np.float64,
)
doublet_predictions = np.full(
    EXPECTED_INPUT_SHAPE[0],
    -1,
    dtype=np.int8,
)
doublet_status = np.full(
    EXPECTED_INPUT_SHAPE[0],
    "not_checked",
    dtype=object,
)

sample_counts = obs_output[SAMPLE_COLUMN].value_counts()
print(sample_counts)
print("Input shape:", observed_shape)
print("Counts encoding:", counts_encoding)
print("Counts nonzero entries:", int(counts_group["data"].shape[0]))

Opening DS3 H5AD with direct HDF5 access
sample
GSM5518597_rGBM-01-B     10960
GSM5518615_rGBM-02-5      9711
GSM5518629_ndGBM-10       9691
GSM5518601_ndGBM-01-C     9470
GSM5518602_ndGBM-01-D     8770
GSM5518613_rGBM-02-3      8727
GSM5518636_ndGBM-07       8627
GSM5518621_rGBM-04-3      7718
GSM5518608_ndGBM-02-1     7436
GSM5518633_ndGBM-04       7335
GSM5518638_LGG-03         6833
GSM5518611_ndGBM-02-5     6613
GSM5518622_rGBM-04-4      6412
GSM5518619_rGBM-04-1      6378
GSM5518634_ndGBM-05       6377
GSM5518620_rGBM-04-2      6209
GSM5518599_rGBM-01-D      5992
GSM5518610_ndGBM-02-4     5739
GSM5518609_ndGBM-02-2     5728
GSM5518598_rGBM-01-C      5653
GSM5518637_ndGBM-08       5650
GSM5518639_ndGBM-09       5110
GSM5518603_ndGBM-01-F     4962
GSM5518600_ndGBM-01-A     4631
GSM5518626_rGBM-05-1      4609
GSM5518596_rGBM-01-A      4324
GSM5518618_rGBM-03-3      3921
GSM5518623_ndGBM-03-1     3574
GSM5518616_rGBM-03-1      3461
GSM5518625_ndGBM-03-3     3090
GSM5518635_ndGBM-06   

In [5]:
scrublet_parameters = {
    "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
    "sim_doublet_ratio": SIM_DOUBLET_RATIO,
    "n_prin_comps": N_PCS,
    "random_seed": RANDOM_SEED,
    "mean_center": False,
    "normalize_variance": True,
    "log_transform": False,
    "use_approx_neighbors": True,
    "simulation_expression_source": "adata.X (layer=None)",
    "simulation_seed_keyword": next(
        key
        for key in ("random_seed", "random_state", "rng")
        if key in simulation_parameters()
    ),
    "core_seed_keyword": next(
        key
        for key in ("random_seed", "random_state", "rng")
        if key in core_scrublet_parameters(N_PCS)
    ),
    "preprocessing": (
        "min_cells=3; min_genes=3; normalize_total; log1p; "
        "scanpy-default highly_variable_genes"
    ),
}
run_rows = []

# Deterministic chunking is retained for the unlikely case that one sample
# exceeds MAX_CELLS_PER_RUN. Current DS3 samples are smaller than this limit.
rng = np.random.default_rng(RANDOM_SEED)

grouped_indices = obs_output.groupby(
    SAMPLE_COLUMN,
    observed=True,
    sort=True,
).indices

try:
    for sample_id, integer_indices in grouped_indices.items():
        sample_positions = np.sort(
            np.asarray(integer_indices, dtype=np.int64)
        )
        n_cells = len(sample_positions)

        print(f"\nSample: {sample_id} | cells: {n_cells}")

        if n_cells < MIN_CELLS_PER_SAMPLE:
            print("Skipping: fewer than 500 cells.")
            run_rows.append(
                {
                    "sample": sample_id,
                    "chunk": "0",
                    "cells": n_cells,
                    "checked_cells": 0,
                    "status": "not_checked_too_few_cells",
                    "predicted_doublets": np.nan,
                    "genes_after_min_cells": np.nan,
                    "highly_variable_genes": np.nan,
                    "n_prin_comps": np.nan,
                    "source": "not_checked",
                    "error": "",
                    "traceback": "",
                }
            )
            continue

        if n_cells > MAX_CELLS_PER_RUN:
            shuffled = sample_positions.copy()
            rng.shuffle(shuffled)
            initial_chunks = np.array_split(
                shuffled,
                int(np.ceil(n_cells / MAX_CELLS_PER_RUN)),
            )
        else:
            initial_chunks = [sample_positions]

        for chunk_number, chunk_positions in enumerate(initial_chunks, start=1):
            chunk_positions = np.sort(
                np.asarray(chunk_positions, dtype=np.int64)
            )
            chunk_label = str(chunk_number)
            cache_path = cache_path_for(sample_id, chunk_label)

            print(
                f"  Running chunk {chunk_number}/{len(initial_chunks)} | "
                f"cells: {len(chunk_positions)}"
            )

            cached = load_chunk_cache(cache_path, chunk_positions)
            if cached is not None:
                positions, scores, predictions, diagnostics = cached
                source = "cache"
                print("    loaded deterministic cached result")
            else:
                try:
                    positions, scores, predictions, diagnostics = (
                        run_one_scrublet_chunk(
                            counts_group=counts_group,
                            obs_names=obs_names,
                            var_names=var_names,
                            row_positions=chunk_positions,
                            sample_id=sample_id,
                            chunk_label=chunk_label,
                        )
                    )
                    save_chunk_cache(
                        cache_path,
                        positions,
                        scores,
                        predictions,
                        diagnostics,
                    )
                    source = "computed"

                except Exception as exc:
                    error_traceback = traceback.format_exc()
                    run_rows.append(
                        {
                            "sample": sample_id,
                            "chunk": chunk_label,
                            "cells": int(len(chunk_positions)),
                            "checked_cells": 0,
                            "status": "failed",
                            "predicted_doublets": np.nan,
                            "genes_after_min_cells": np.nan,
                            "highly_variable_genes": np.nan,
                            "n_prin_comps": np.nan,
                            "source": "computed",
                            "error": repr(exc),
                            "traceback": error_traceback,
                        }
                    )

                    pd.DataFrame(run_rows).to_csv(
                        AUDIT_DIR / "GSE182109_scrublet_run_log.tsv",
                        sep="\t",
                        index=False,
                    )

                    raise RuntimeError(
                        "Scrublet stopped at the first failed sample so the "
                        "remaining samples are not wasted. See the final row "
                        "of GSE182109_scrublet_run_log.tsv for the full "
                        f"traceback. Sample={sample_id!r}, chunk={chunk_label}."
                    ) from exc

            doublet_scores[positions] = scores
            doublet_predictions[positions] = predictions.astype(np.int8)
            doublet_status[positions] = "checked"

            run_rows.append(
                {
                    "sample": sample_id,
                    "chunk": chunk_label,
                    "cells": int(len(chunk_positions)),
                    "checked_cells": int(len(positions)),
                    "status": "checked",
                    "predicted_doublets": int(predictions.sum()),
                    "genes_after_min_cells": diagnostics[
                        "genes_after_min_cells"
                    ],
                    "highly_variable_genes": diagnostics[
                        "highly_variable_genes"
                    ],
                    "n_prin_comps": diagnostics["n_prin_comps"],
                    "source": source,
                    "error": "",
                    "traceback": "",
                }
            )

            pd.DataFrame(run_rows).to_csv(
                AUDIT_DIR / "GSE182109_scrublet_run_log.tsv",
                sep="\t",
                index=False,
            )

            print(
                "    checked:",
                len(positions),
                "| HVGs:",
                diagnostics["highly_variable_genes"],
                "| predicted doublets:",
                int(predictions.sum()),
            )
            gc.collect()

finally:
    input_h5.close()
    del counts_group
    gc.collect()


run_table = pd.DataFrame(run_rows)
run_table.to_csv(
    AUDIT_DIR / "GSE182109_scrublet_run_log.tsv",
    sep="\t",
    index=False,
)

obs_output["doublet_score"] = doublet_scores

predicted_series = pd.Series(
    pd.array(
        [pd.NA] * len(obs_output),
        dtype="boolean",
    ),
    index=obs_output.index,
)

checked_mask = doublet_predictions >= 0
predicted_series.iloc[np.flatnonzero(checked_mask)] = (
    doublet_predictions[checked_mask].astype(bool)
)

obs_output["predicted_doublet"] = predicted_series
obs_output["doublet_check_status"] = pd.Categorical(
    doublet_status,
    categories=["not_checked", "checked", "failed"],
)

print()
print("All sample chunks completed.")



Sample: GSM5518596_rGBM-01-A | cells: 4324
  Running chunk 1/1 | cells: 4324
    checked: 4324 | HVGs: 2487 | predicted doublets: 1

Sample: GSM5518597_rGBM-01-B | cells: 10960
  Running chunk 1/1 | cells: 10960
    checked: 10960 | HVGs: 1567 | predicted doublets: 1

Sample: GSM5518598_rGBM-01-C | cells: 5653
  Running chunk 1/1 | cells: 5653
    checked: 5653 | HVGs: 1518 | predicted doublets: 1

Sample: GSM5518599_rGBM-01-D | cells: 5992
  Running chunk 1/1 | cells: 5992
    checked: 5992 | HVGs: 2539 | predicted doublets: 45

Sample: GSM5518600_ndGBM-01-A | cells: 4631
  Running chunk 1/1 | cells: 4631
    checked: 4631 | HVGs: 1867 | predicted doublets: 16

Sample: GSM5518601_ndGBM-01-C | cells: 9470
  Running chunk 1/1 | cells: 9470
    checked: 9470 | HVGs: 1435 | predicted doublets: 0

Sample: GSM5518602_ndGBM-01-D | cells: 8770
  Running chunk 1/1 | cells: 8770
    checked: 8770 | HVGs: 2040 | predicted doublets: 199

Sample: GSM5518603_ndGBM-01-F | cells: 4962
  Running chun

In [6]:
predicted_doublet = obs_output["predicted_doublet"]

n_doublets = int((predicted_doublet == True).sum())
n_singlets = int((predicted_doublet == False).sum())
n_not_checked = int(predicted_doublet.isna().sum())

print("Predicted doublets:", n_doublets)
print("Predicted singlets:", n_singlets)
print("Not checked:", n_not_checked)
print(obs_output["doublet_score"].describe())

historical_comparison = {
    "observed_predicted_doublets": n_doublets,
    "historical_predicted_doublets": EXPECTED_PREDICTED_DOUBLETS,
    "observed_predicted_singlets": n_singlets,
    "historical_predicted_singlets": EXPECTED_PREDICTED_SINGLETS,
    "observed_not_checked": n_not_checked,
    "historical_not_checked": EXPECTED_NOT_CHECKED,
}

if (
    n_doublets != EXPECTED_PREDICTED_DOUBLETS
    or n_singlets != EXPECTED_PREDICTED_SINGLETS
    or n_not_checked != EXPECTED_NOT_CHECKED
):
    print(
        "WARNING: Scrublet calls differ from the historical run. "
        "The sparse memory patch and/or automatic fallback chunking can "
        "change calls slightly. Review package versions and the run log."
    )

historical_comparison

Predicted doublets: 3524
Predicted singlets: 214549
Not checked: 662
count    218073.000000
mean          0.053088
std           0.055176
min           0.000566
25%           0.020179
50%           0.036713
75%           0.066474
max           0.698413
Name: doublet_score, dtype: float64


{'observed_predicted_doublets': 3524,
 'historical_predicted_doublets': 2927,
 'observed_predicted_singlets': 214549,
 'historical_predicted_singlets': 215146,
 'observed_not_checked': 662,
 'historical_not_checked': 662}

In [7]:
per_sample_summary = (
    obs_output.groupby(SAMPLE_COLUMN, observed=True)
    .agg(
        n_cells=("predicted_doublet", "size"),
        n_doublets=(
            "predicted_doublet",
            lambda values: int((values == True).sum()),
        ),
        n_singlets=(
            "predicted_doublet",
            lambda values: int((values == False).sum()),
        ),
        n_not_checked=(
            "predicted_doublet",
            lambda values: int(values.isna().sum()),
        ),
        mean_doublet_score=("doublet_score", "mean"),
        median_doublet_score=("doublet_score", "median"),
        maximum_doublet_score=("doublet_score", "max"),
    )
    .reset_index()
)

checked_denominator = (
    per_sample_summary["n_doublets"]
    + per_sample_summary["n_singlets"]
)
per_sample_summary["doublet_percent_checked"] = np.where(
    checked_denominator > 0,
    100
    * per_sample_summary["n_doublets"]
    / checked_denominator,
    np.nan,
)

per_sample_summary.to_csv(
    AUDIT_DIR / "GSE182109_scrublet_summary_by_sample.tsv",
    sep="\t",
    index=False,
)

obs_output[
    [
        SAMPLE_COLUMN,
        "doublet_score",
        "predicted_doublet",
        "doublet_check_status",
    ]
].to_csv(
    AUDIT_DIR / "GSE182109_scrublet_per_cell_scores.tsv.gz",
    sep="\t",
    compression="gzip",
)

per_sample_summary.sort_values(
    "doublet_percent_checked",
    ascending=False,
)

,sample,n_cells,n_doublets,n_singlets,n_not_checked,mean_doublet_score,median_doublet_score,maximum_doublet_score,doublet_percent_checked
19,GSM5518615_rGBM-02-5,9711,974,8737,0,0.051720,0.031250,0.451923,10.029863
37,GSM5518633_ndGBM-04,7335,275,7060,0,0.044805,0.027828,0.598592,3.749148
23,GSM5518619_rGBM-04-1,6378,231,6147,0,0.042580,0.025187,0.580882,3.621825
38,GSM5518634_ndGBM-05,6377,217,6160,0,0.041389,0.026565,0.580882,3.402854
42,GSM5518638_LGG-03,6833,229,6604,0,0.048954,0.031373,0.683333,3.351383
34,GSM5518630_LGG-04-1,2916,72,2844,0,0.044591,0.035385,0.586957,2.469136
14,GSM5518610_ndGBM-02-4,5739,131,5608,0,0.039387,0.024225,0.568182,2.282628
6,GSM5518602_ndGBM-01-D,8770,199,8571,0,0.060909,0.048057,0.547619,2.269099
26,GSM5518622_rGBM-04-4,6412,128,6284,0,0.039556,0.025187,0.580882,1.996257
12,GSM5518608_ndGBM-02-1,7436,148,7288,0,0.044039,0.030612,0.598592,1.990317


In [8]:
checked_scores = (
    obs_output["doublet_score"]
    .dropna()
    .to_numpy(dtype=np.float64)
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(checked_scores, bins=80, edgecolor="black")
ax.set_xlabel("Scrublet doublet score")
ax.set_ylabel("Cells")
ax.set_title("GSE182109 post-QC Scrublet scores")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "GSE182109_scrublet_score_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

In [9]:
doublet_summary = {
    "dataset_id": "DS3_GSE182109",
    "geo_accession": "GSE182109",
    "scope": "sample-aware post-QC diagnostic",
    "method": "scanpy.pp.scrublet",
    "sample_column": SAMPLE_COLUMN,
    "parameters": scrublet_parameters,
    "minimum_cells_per_sample": MIN_CELLS_PER_SAMPLE,
    "maximum_cells_per_initial_run": MAX_CELLS_PER_RUN,
    "input_cells": int(len(obs_output)),
    "input_genes": int(len(var_names)),
    "predicted_doublets": n_doublets,
    "predicted_singlets": n_singlets,
    "not_checked": n_not_checked,
    "doublets_removed_from_main_analysis": False,
    "main_analysis_cells": int(len(obs_output)),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
    "runtime_patch": (
        "direct HDF5 CSR row reads; memory-bounded pre-HVG filtering; "
        "sample-wise advanced Scanpy Scrublet; simulation from adata.X; "
        "mean_center=False; deterministic per-sample cache; "
        "metadata-only H5AD patch"
    ),
}

uns_output["doublet_detection_provenance"] = {
    "scope": "sample_aware_post_QC_diagnostic",
    "method": "scanpy.pp.scrublet",
    "doublets_removed_from_main_analysis": "false",
    "predicted_doublets": n_doublets,
    "predicted_singlets": n_singlets,
    "not_checked": n_not_checked,
    "parameters_json": json.dumps(
        scrublet_parameters,
        sort_keys=True,
    ),
    "runtime_patch": doublet_summary["runtime_patch"],
}

with open(
    AUDIT_DIR / "GSE182109_scrublet_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(doublet_summary, handle, indent=2)

print(doublet_summary)


{'dataset_id': 'DS3_GSE182109', 'geo_accession': 'GSE182109', 'scope': 'sample-aware post-QC diagnostic', 'method': 'scanpy.pp.scrublet', 'sample_column': 'sample', 'parameters': {'expected_doublet_rate': 0.05, 'sim_doublet_ratio': 1.0, 'n_prin_comps': 20, 'random_seed': 0, 'mean_center': False, 'normalize_variance': True, 'log_transform': False, 'use_approx_neighbors': True, 'simulation_expression_source': 'adata.X (layer=None)', 'simulation_seed_keyword': 'random_seed', 'core_seed_keyword': 'random_state', 'preprocessing': 'min_cells=3; min_genes=3; normalize_total; log1p; scanpy-default highly_variable_genes'}, 'minimum_cells_per_sample': 500, 'maximum_cells_per_initial_run': 40000, 'input_cells': 218735, 'input_genes': 30284, 'predicted_doublets': 3524, 'predicted_singlets': 214549, 'not_checked': 662, 'doublets_removed_from_main_analysis': False, 'main_analysis_cells': 218735, 'timestamp': '2026-08-05T07:17:50', 'output_h5ad': 'G:\\Repository\\glioma-cnv-single-cell-atlas\\data\\p

In [10]:
if (len(obs_output), len(var_names)) != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        "The doublet diagnostic unexpectedly changed the object shape: "
        f"{(len(obs_output), len(var_names))}"
    )

temporary_output = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.temporary{OUTPUT_H5AD.suffix}"
)
metadata_sidecar = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.metadata_sidecar{OUTPUT_H5AD.suffix}"
)

for path in (temporary_output, metadata_sidecar):
    if path.exists():
        path.unlink()

input_size = INPUT_H5AD.stat().st_size
free_space = shutil.disk_usage(OUTPUT_H5AD.parent).free
required_free_space = input_size + 2 * 1024**3

if free_space < required_free_space:
    raise OSError(
        "Insufficient free disk space for the atomic output copy. "
        f"Input size: {input_size / 1024**3:.2f} GiB; "
        f"free space: {free_space / 1024**3:.2f} GiB; "
        f"required: {required_free_space / 1024**3:.2f} GiB."
    )

print("Copying the large input H5AD on disk...")
shutil.copy2(
    INPUT_H5AD,
    temporary_output,
)

if not temporary_output.exists():
    raise FileNotFoundError(
        f"Temporary output was not created: {temporary_output}"
    )


# Encode only the updated metadata in a tiny sidecar H5AD.
metadata_holder = ad.AnnData(
    X=sp.csr_matrix(
        (len(obs_output), 0),
        dtype=np.float32,
    ),
    obs=obs_output,
    var=pd.DataFrame(index=pd.Index([], dtype=str)),
    uns=uns_output,
)

metadata_holder.write_h5ad(
    metadata_sidecar,
    compression="lzf",
)

del metadata_holder
gc.collect()


try:
    # Replace obs and uns inside the copied file without reading X or layers.
    with h5py.File(metadata_sidecar, "r") as source_handle:
        with h5py.File(temporary_output, "r+") as output_handle:
            for group_name in ("obs", "uns"):
                if group_name not in source_handle:
                    raise AssertionError(
                        f"Metadata sidecar is missing /{group_name}."
                    )

                if group_name in output_handle:
                    del output_handle[group_name]

                source_handle.copy(
                    group_name,
                    output_handle,
                    name=group_name,
                )

            output_handle.flush()

    # Validate only metadata and HDF5 matrix descriptors. Calling
    # sc.read_h5ad(..., backed='r') would still load the sparse counts layer
    # in this AnnData version and reproduce the 2.13-GiB allocation failure.
    with h5py.File(temporary_output, "r") as check_handle:
        if "X" not in check_handle:
            raise AssertionError("Saved output is missing /X.")
        if stored_matrix_shape(check_handle["X"]) != EXPECTED_INPUT_SHAPE:
            raise AssertionError(
                "Saved-file validation failed: "
                f"observed {stored_matrix_shape(check_handle['X'])}"
            )

        if "layers" not in check_handle or "counts" not in check_handle["layers"]:
            raise AssertionError("Saved output is missing layers['counts'].")
        if stored_matrix_shape(check_handle["layers"]["counts"]) != EXPECTED_INPUT_SHAPE:
            raise AssertionError(
                "Saved counts layer has an unexpected shape."
            )

        saved_obs = read_elem(check_handle["obs"])
        saved_uns = read_elem(check_handle["uns"])

    for column in (
        "doublet_score",
        "predicted_doublet",
        "doublet_check_status",
    ):
        if column not in saved_obs.columns:
            raise AssertionError(
                f"Saved output is missing obs[{column!r}]."
            )

    if "doublet_detection_provenance" not in saved_uns:
        raise AssertionError(
            "Saved output is missing "
            "uns['doublet_detection_provenance']."
        )

    saved_predictions = saved_obs["predicted_doublet"]

    if int((saved_predictions == True).sum()) != n_doublets:
        raise AssertionError(
            "Saved predicted-doublet count does not match the in-memory result."
        )
    if int((saved_predictions == False).sum()) != n_singlets:
        raise AssertionError(
            "Saved predicted-singlet count does not match the in-memory result."
        )
    if int(saved_predictions.isna().sum()) != n_not_checked:
        raise AssertionError(
            "Saved not-checked count does not match the in-memory result."
        )

    temporary_output.replace(OUTPUT_H5AD)

finally:
    if metadata_sidecar.exists():
        metadata_sidecar.unlink()


print()
print("Saved and validated:", OUTPUT_H5AD)
print("Validated shape:", EXPECTED_INPUT_SHAPE)
print("Validated layer: counts")
print(
    "Validated obs columns: doublet_score, predicted_doublet, "
    "doublet_check_status"
)
print("Validated uns entry: doublet_detection_provenance")

Copying the large input H5AD on disk...

Saved and validated: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS3_GSE182109\GSE182109_postQC_scrublet_checked.h5ad
Validated shape: (218735, 30284)
Validated layer: counts
Validated obs columns: doublet_score, predicted_doublet, doublet_check_status
Validated uns entry: doublet_detection_provenance
